# DRM · Lab 05 — Assignment 04 Sheet (GRADED)
## Options Strategy Performance Back-Test on **your selected Nifty 50 stock**

**Course:** DRM (<ECON F354>) · <Semester 1 - 2026-27>
**Instructor:** <Prof. Aditya Sharma> · BITS Pilani
**Total marks:** 15 **Deadline:** `<27 NOV 2026, Midnight>`

---

### What this assignment is

A systematic two-year comparison of **three option strategies** on the stock allotted to you in
Assignment 03:

1. **Covered call** = long stock + short out-of-the-money call
2. **Protective put** = long stock + long at-the-money put
3. **Long straddle** = long at-the-money call + long at-the-money put

Each strategy starts with **₹1,00,000**, is rebalanced **monthly (every 30 days)** and is compared
with the Nifty 50 and with simply holding the stock.

### Before you start

1. Complete **Assignment 03** first. You will reuse your own `bsm_price` function from it.
2. Fill in the identification cell below, **with the same ticker as in Assignment 03**.
   **A submission without it will not be graded.**
3. Write your code in the cells marked `# ---- YOUR CODE HERE ----` and your written answers in
   the markdown cells marked `**Your answer:**`. Do not delete the question cells.
4. **Marks are for interpretation as much as for code.** A correct plot with no commentary earns
   roughly half the marks for that part. Every question that asks "why" or "comment" is looking
   for economics, not for a restatement of what the chart shows.
5. The rubric for each question is printed **directly under that question**.

### Submission

- `Kernel → Restart & Run All`, confirm there are **no errors and all outputs are visible**.
- Save as **`DRM_Lab05_<CampusID>.ipynb`** — e.g. `DRM_Lab05_2023A7PS0123P.ipynb`.
- Upload to the Lab 05 assignment on **Nalanda**. Do not email it; do not open a pull request.

### Marks

| Q | Topic | Marks |
|---|---|---|
| Q1 | Data, pricer check and rebalancing calendar | 2 |
| Q2 | Strategy engine and monthly log | 4 |
| Q3 | Performance metrics | 2 |
| Q4 | Charts: cumulative returns, drawdowns, histograms | 2 |
| Q5 | Market-regime analysis: bull, bear, sideways | 2 |
| Q6 | Sensitivity and interpretation | 3 |
| | **Total** | **15** |

---
## Evaluation rubric

Each question below carries its own **criterion table** showing how its marks are split. Every
criterion in those tables is graded on the same four-level scale:

| Level | Share of the criterion's marks | Code and output | Written interpretation |
|---|---|---|---|
| **Complete** | 100% | Runs on `Restart & Run All`; result is correct; the required check (assertion, benchmark, table) is present and passes; figure has a title, axis labels with units and a legend | Explains the **mechanism** in economic terms, refers to the student's **own numbers**, and states data provenance (live or simulated) |
| **Substantial** | about 70% | Correct method with one minor slip (a unit, a day-count, an unlabelled axis, a missing check) | Right conclusion, but the mechanism is thin or the numbers are not quoted |
| **Partial** | about 40% | Runs, but a formula or convention is wrong, or part of the required output is missing | Restates what the chart shows without explaining why; or generic text that could belong to any stock |
| **None** | 0% | Missing, does not run, or hard-coded output | Blank, or copied question text |

**How the levels turn into marks.** A criterion worth 2 marks graded *Substantial* earns about
1.5; graded *Partial*, about 1. Marks are awarded in steps of 0.5.

**Notebook-wide rules** (applied after the question marks are added up):

| Notebook-wide issue | Effect |
|---|---|
| Identification cell or honour statement not completed | Not graded |
| `Restart & Run All` raises an error | Cells after the first error are graded at most at the **Partial** level |
| Outputs not visible in the saved file | The affected part is graded as **None** |
| Simulated (`[SIM ]`) or model-generated numbers presented as market data | −2 marks |
| A required source is missing (each of the look-ups listed in the checklist) | −1 mark each |
| Analysis run on a stock other than the one allotted | Not graded |
| AI or outside help used but not disclosed in the declaration | Referred as an integrity violation |
| Late submission | `<late-submission policy>` |

**A back-test is only as good as its conventions.** Two submissions can both be "correct" and
report different numbers if they size positions differently. The conventions in Q2 are therefore
**fixed**; departures from them are graded as errors unless you state and justify them.

In [ ]:
# ==================================================================================
#  IDENTIFICATION — fill this in. Submissions without it will not be graded.
# ==================================================================================
NAME       = ""          # e.g. "Ananya Sharma"
CAMPUS_ID  = ""          # e.g. "2023A7PS0123P"
EMAIL      = ""

# The Nifty 50 company allotted to you. Everything below is computed on THIS stock.
TICKER     = ""          # the SAME ticker as in Assignment 03, e.g. "HDFCBANK.NS"

# Honour statement. Type "I agree" to confirm that the work below is your own and that any
# outside help, source or AI assistance is disclosed in the final cell of this notebook.
HONOUR     = ""

assert NAME and CAMPUS_ID and TICKER and HONOUR.strip().lower() == "i agree", \
    "Fill in NAME, CAMPUS_ID, TICKER and type 'I agree' in HONOUR before running the rest."
print(f"{NAME} | {CAMPUS_ID} | {TICKER}")

---
## Provided setup — run these two cells as-is

The data layer tries the live API first and falls back to a seeded simulator. **If your loaders
report `[SIM ]`, say so in your write-up** — you will not be penalised for working offline, but you
will be penalised for presenting simulated output as though it were market data.

`stock.loc[WIN_START:]` is the two-year back-test window; about four extra months are downloaded
before it so that rolling volatility is defined on the first rebalancing date. `nifty` is aligned
to the stock's trading calendar.

In [ ]:
# !pip install -q yfinance pandas_datareader        # uncomment in Colab if needed

import warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (11, 4.4), "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "legend.frameon": False,
})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

SEED      = 2026
TODAY     = dt.date.today()
END       = TODAY.isoformat()
WIN_START = (TODAY - dt.timedelta(days=730)).isoformat()          # start of the 2-year analysis window
START     = (TODAY - dt.timedelta(days=730 + 120)).isoformat()    # + ~4 months of warm-up for rolling volatility
DATA_MODE = "auto"          # "auto" | "live" | "offline"
IS_SIM = {}

def ttl(title, key=None):
    return title + ("   —   [SIMULATED DATA]" if IS_SIM.get(key) else "")

print(f"pandas {pd.__version__} | download {START} -> {END} | analysis window from {WIN_START} | DATA_MODE = {DATA_MODE}")

CAPITAL = 100_000.0         # INR invested in each strategy at the start

In [ ]:
def fetch_yahoo(ticker, start=START, end=END):
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if df is None or len(df) == 0:
        raise RuntimeError("empty frame")
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)
    s = df["Close"].astype(float).dropna()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.rename(ticker)

def fetch_fred(series, start=START, end=END):
    from pandas_datareader import data as pdr
    s = pdr.DataReader(series, "fred", start, end).iloc[:, 0].astype(float).dropna()
    s.index = pd.to_datetime(s.index)
    return s.rename(series)

def sim_gbm_series(name, S0, mu, sigma, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt = 1 / 252
    incr = (mu - 0.5 * sigma**2) * dtt + sigma * np.sqrt(dtt) * rng.standard_normal(len(idx))
    return pd.Series(S0 * np.exp(np.cumsum(incr)), index=idx, name=name)

def sim_ou_series(name, x0, level, kappa, vol, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt, x, out = 1 / 252, x0, np.empty(len(idx))
    for i in range(len(idx)):
        x += kappa * (level - x) * dtt + vol * np.sqrt(dtt) * rng.standard_normal()
        out[i] = x
    return pd.Series(out, index=idx, name=name)

SIM_ANCHOR = "2021-01-01"      # simulated paths start here, so they do not change from one day to the next

def sim_window(s, S0=None):
    """Cut a simulated path to the download window; optionally rescale it to start at S0."""
    s = s.loc[START:]
    return s if S0 is None else (s / s.iloc[0] * S0).rename(s.name)

def load(key, live_fn, sim_fn):
    if DATA_MODE != "offline":
        try:
            s = live_fn(); IS_SIM[key] = False
            print(f"  [LIVE] {key:<10s} {len(s):>5d} obs  {s.index.min().date()} -> {s.index.max().date()}")
            return s
        except Exception as e:
            if DATA_MODE == "live":
                raise
            print(f"  [SIM ] {key:<10s} live fetch failed ({type(e).__name__}) -> simulated")
    else:
        print(f"  [SIM ] {key:<10s} DATA_MODE=offline")
    IS_SIM[key] = True
    return sim_fn()

def sim_stock_from_market(name, market, S0, beta, idio_vol, alpha, seed):
    """Offline stand-in for a stock: beta x market log-return + idiosyncratic noise."""
    full = pd.bdate_range(SIM_ANCHOR, END)
    e = pd.Series(idio_vol * np.sqrt(1 / 252) * np.random.default_rng(seed).standard_normal(len(full)), index=full)
    m = np.log(market).diff().fillna(0.0)
    path = np.exp((alpha / 252 + beta * m + e.reindex(m.index).fillna(0.0)).cumsum())
    return (path / path.iloc[0] * S0).rename(name)

print("Loading\n" + "-" * 70)
nifty = load("nifty", lambda: fetch_yahoo("^NSEI"),
                      lambda: sim_window(sim_gbm_series("^NSEI", 1.0, 0.11, 0.140, SEED + 94, start=SIM_ANCHOR), 23_500.0))
stock = load("stock", lambda: fetch_yahoo(TICKER),
                      lambda: sim_stock_from_market(TICKER, nifty, 1_450.0, 1.10, 0.180, 0.01, SEED + 98))
inr_r = load("inr_r", lambda: fetch_fred("INDIRLTLT01STM") / 100.0,
                      lambda: sim_window(sim_ou_series("INR rate", 0.066, 0.066, 1.0, 0.007, SEED + 74, start=SIM_ANCHOR)))
print("-" * 70)

nifty = nifty.reindex(stock.index).ffill().bfill()
inr_r = inr_r.reindex(stock.index).ffill().bfill()
print(f"{TICKER} {stock.iloc[-1]:,.2f} | NIFTY {nifty.iloc[-1]:,.2f} | r_INR {inr_r.iloc[-1]:.3%} | "
      f"{len(stock.loc[WIN_START:])} trading days in the 2-year window")

---
## Q1 · Data, pricer check and rebalancing calendar · **[4 marks]**

**(a)** Paste your `bsm_price(S, K, T, r, sigma, q=0, kind="call")` from Assignment 03 into the
cell below. The provided test cell that follows must pass **unchanged**.

**(b)** Compute daily log returns and the **21-day rolling annualised volatility** `rv21` of your
stock. Report the two-year annualised volatility of the stock and of the Nifty, their correlation,
and the stock's **beta** to the Nifty. Set the dividend yield `Q_DIV` you cited in Assignment 03.

**(c)** Write `rebalance_dates(index, start, step_days=30)`: the first date is the first trading
day on or after `start`; each subsequent date is the first trading day on or after the previous
date + 30 calendar days; stop when the next date would fall beyond the data. Print the number of
**complete cycles** (it should be about 24) and the first and last three dates. **Assert** that
every gap between consecutive dates is between 30 and 35 calendar days.

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Own pricer pasted; the provided test cell passes without modification | 1 |
| (b) | Volatilities, correlation and beta computed from log returns on the 2-year window; `rv21` uses only past data; `Q_DIV` set with its source restated | 1 |
| (c) | Calendar correct at both ends (no cycle whose exit date is missing); count reported; gap assertion present and passing | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q1 (a): paste your pricer from Assignment 03

def bsm_price(S, K, T, r, sigma, q=0.0, kind="call"):
    # TODO
    raise NotImplementedError

In [ ]:
# ---- PROVIDED TEST — do not edit. It must pass before you go on. ----
assert abs(bsm_price(42, 40, 0.5, 0.10, 0.20, kind="call") - 4.76) < 0.005, "call benchmark failed"
assert abs(bsm_price(42, 40, 0.5, 0.10, 0.20, kind="put")  - 0.81) < 0.005, "put benchmark failed"
_c = bsm_price(1500.0, 1575.0, 30 / 365, 0.065, 0.25, 0.01, "call")
_p = bsm_price(1500.0, 1575.0, 30 / 365, 0.065, 0.25, 0.01, "put")
assert abs(_c - _p - (1500.0 * np.exp(-0.01 * 30 / 365) - 1575.0 * np.exp(-0.065 * 30 / 365))) < 1e-8, "parity failed"
assert 0 < _c < _p, "a 5% OTM call must be worth less than the 5% ITM put"
print("bsm_price passes the benchmark and parity tests.")

In [ ]:
# ---- YOUR CODE HERE ----  Q1 (b) and (c)
Q_DIV        = None     # TODO: dividend yield of your stock (decimal), as cited in Assignment 03
Q_DIV_SOURCE = ""

# TODO: log returns, rv21, 2-year vols of stock and Nifty, correlation, beta

def rebalance_dates(index, start, step_days=30):
    # TODO
    raise NotImplementedError

# TODO: print the number of complete cycles, first/last three dates; assert the gaps are 30-35 days

---
## Q2 · Strategy engine and monthly log · **[4 marks]**

On each rebalancing date $t_0$ a new position is opened in each strategy and held to the next
rebalancing date $t_1$, which is also the options' expiry. Let $S_0, S_1$ be the closes on those
dates, $T = (t_1 - t_0)/365$, $r$ the rate at $t_0$, and $\sigma$ = `rv21` at $t_0$ **plus**
`VOL_PREMIUM` (set to 0 in the base case). All option premiums are computed with **your**
`bsm_price`; there are no transaction costs in the base case.

**Fixed conventions — use exactly these, so that everyone's numbers are comparable.**

| Strategy | Position per unit | Capital per unit at $t_0$ | Value per unit at $t_1$ | Return for the cycle |
|---|---|---|---|---|
| **Covered call** | +1 share, −1 call at $K_c = 1.05\,S_0$ | $S_0 - C_0(K_c)$ | $\min(S_1, K_c)$ | $\dfrac{\min(S_1, K_c)}{S_0 - C_0(K_c)} - 1$ |
| **Protective put** | +1 share, +1 put at $K = S_0$ | $S_0 + P_0$ | $\max(S_1, S_0)$ | $\dfrac{\max(S_1, S_0)}{S_0 + P_0} - 1$ |
| **Long straddle** | +1 call, +1 put at $K = S_0$, **on a notional equal to the capital**; the cash not spent on premium earns $r$ | $S_0$ (of which $\pi = C_0 + P_0$ is premium) | $\lvert S_1 - S_0\rvert + (S_0 - \pi)e^{rT}$ | $\dfrac{\lvert S_1 - S_0\rvert - \pi}{S_0} + \Big(1 - \dfrac{\pi}{S_0}\Big)\big(e^{rT} - 1\big)$ |

- **Capital compounds**: the whole of each strategy's capital is redeployed at every rebalancing
  date, so the value after month $m$ is ₹1,00,000 $\times \prod_{i \le m}(1 + R_i)$.
- **Fractional units** are allowed (capital ÷ capital-per-unit); lot sizes are ignored.
- **Dividends** on the stock leg are ignored in the returns (closes are unadjusted) but $q$ enters
  the option premiums.
- **Benchmarks**: the Nifty 50 and the stock itself, bought and held over the same cycles.

**(a)** Write `run_backtest(...)` returning one row per cycle with: entry and exit dates, $S_0$,
$S_1$, $T$, $r$, $\sigma$, the three premiums $C_0(K_c)$, $C_0(S_0)$, $P_0(S_0)$, and the cycle
return of each strategy, of the stock and of the Nifty. Give it keyword arguments `vol_premium=0.0`,
`call_otm=0.05` and `straddle_sizing="notional"` — you will change them in Q6.

**(b)** Display the **monthly log** in this layout (all cycles), with returns in percent:

| Month | Covered Call | Protective Put | Long Straddle | Stock | Nifty Return |
|---|---|---|---|---|---|
| Jan-25 | +3.2% | −2.1% | +8.5% | +4.5% | +4.1% |

**(c)** Include **two assertions** that test your engine against known bounds on every cycle:
the covered-call return can never exceed $K_c/(S_0 - C_0) - 1$, and the protective-put return can
never be below $S_0/(S_0 + P_0) - 1$.

**(d)** In the markdown cell below:
- For each strategy, state in one line what it is **long or short** of in terms of direction and
  volatility, and the best-case and worst-case return for a single cycle.
- The straddle is sized on **notional**, not by spending the whole ₹1,00,000 on premium. What would
  the alternative do to its monthly returns, and why is the notional convention the fairer
  comparison with the two stock-based strategies?
- Where exactly could **look-ahead bias** enter this back-test, and how does your code avoid it?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Covered-call return correct (strike 5% OTM, premium reduces capital, upside capped) | 1 |
| (a) | Protective-put return correct (ATM put, premium adds to capital, floor at $S_0$) | 1 |
| (a) | Straddle return correct under the notional convention, including interest on idle cash | 1 |
| (a) | Engine uses entry-date volatility and rate only; $T$ from actual calendar days; benchmarks over identical cycles; keyword arguments present | 2 |
| (b), (c) | Monthly log in the required layout; both bound assertions present and passing | 1 |
| (d) | Correct long/short direction-and-volatility description and bounds for all three; leverage argument for notional sizing; look-ahead risks identified (volatility window, exit price, regime labels) | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (a), (b), (c)

def run_backtest(spot, bench, rate, q, vol_premium=0.0, call_otm=0.05, straddle_sizing="notional",
                 vol_window=21, start=WIN_START):
    # TODO: one row per cycle; see the conventions table above
    raise NotImplementedError

# TODO: log = run_backtest(stock, nifty, inr_r, Q_DIV)
# TODO: assertions on the covered-call cap and the protective-put floor
# TODO: display the monthly log

**Your answer (Q2d):**

*Data provenance (live or simulated; option premiums observed or model-generated):*

*Covered call — exposure, best case, worst case:*

*Protective put — exposure, best case, worst case:*

*Long straddle — exposure, best case, worst case:*

*Notional vs all-in sizing of the straddle:*

*Where look-ahead bias could enter, and how you avoided it:*

---
## Q3 · Performance metrics · **[2 marks]**

**(a)** Write `performance(returns)` and report, for each of the three strategies **and** both
benchmarks:

| Metric | Definition |
|---|---|
| Total return | $\prod(1+R_i) - 1$ over all cycles |
| Final value | ₹1,00,000 × (1 + total return) |
| Win rate (%) | share of cycles with $R_i > 0$ |
| Max drawdown | largest peak-to-trough fall of the compounded value, starting from ₹1,00,000 |
| Best month / worst month | $\max R_i$ and $\min R_i$, with the month |
| Volatility of returns | standard deviation of $R_i$, per cycle and annualised ($\times\sqrt{365/30}$) |
| Return ÷ risk | annualised mean excess return over $r$ ÷ annualised volatility |

**(b)** In the markdown cell below: rank the strategies on total return and on return ÷ risk. Do
the two rankings agree? Which strategy has the highest **win rate**, and is a high win rate the
same thing as a good strategy? Use your own best-month and worst-month figures in the answer.

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | All metrics present and correct for five series; drawdown computed on the compounded value including the starting point; annualisation stated | 3 |
| (b) | Rankings stated with own numbers; explains why win rate and average outcome diverge (many small gains against a few large losses for the covered call; the reverse for the straddle) | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q3

def performance(returns, rf_per_cycle=0.0):
    """returns: Series of per-cycle returns. Return a dict of the metrics listed above."""
    # TODO
    raise NotImplementedError

**Your answer (Q3b):**

*Ranking by total return, and by return ÷ risk:*

*Win rate vs quality of a strategy, with your best/worst month figures:*

---
## Q4 · Charts: cumulative returns, drawdowns, histograms · **[2 marks]**

**(a) Cumulative returns chart** — the value of ₹1,00,000 in each of the three strategies and the
two benchmarks, on one set of axes, starting from ₹1,00,000 on the first rebalancing date.

**(b) Drawdown comparison graph** — the percentage drawdown of each of the five series through
time, on one set of axes.

**(c) Histograms of monthly returns** — one panel per strategy, with the **same x-axis range**
across panels, and with the mean and the stock's corresponding distribution marked.

**(d)** In the markdown cell below, describe the **shape** of each strategy's return distribution
(skewness, where the mass is, where the tails are cut off) and connect each shape to the payoff
diagram of the strategy. Which strategy would a pension fund holding the stock prefer, and which
would a trader expecting an earnings surprise prefer?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Five series, common start at ₹1,00,000, labelled, simulated-data flag if applicable | 1 |
| (b) | Drawdowns correct (never positive), comparable on one axis | 1 |
| (c) | Three histograms on a common x-range with mean marked | 1 |
| (d) | Truncated right tail (covered call), truncated left tail (protective put), mass at a small loss with long right tail (straddle), each tied to its payoff; sensible matching of strategy to investor | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (a), (b), (c)

**Your answer (Q4d):**

*Covered call — shape and why:*

*Protective put — shape and why:*

*Long straddle — shape and why:*

*Which investor prefers which:*

---
## Q5 · Market-regime analysis: bull, bear, sideways · **[2 marks]**

**(a)** Classify every cycle by the **Nifty's** return over that cycle:

- **Bull** — Nifty return above `BULL_THR` = +2%
- **Bear** — Nifty return below `BEAR_THR` = −2%
- **Sideways** — in between

You may change the two thresholds if you **justify** the change (for example, to get at least four
cycles in each regime). Report the number of cycles in each regime.

**(b)** Produce a table of the **mean return, win rate and worst month** of each strategy and of
the stock **in each regime**, and a grouped bar chart of mean return by regime and strategy.
Mark on the table which strategy is best in each regime.

**(c)** Repeat the classification using the **stock's own** return instead of the Nifty's and show
the mean-return table again. Report how many cycles change regime.

**(d)** In the markdown cell below:
- Which strategy wins in each regime, and is that what the payoff diagrams predict?
- How different are the Nifty-based and stock-based tables? Explain the difference using your
  stock's **beta and correlation** from Q1, and say which classification is more useful to someone
  deciding which strategy to run on this stock.
- A regime label is known only **after** the month is over. What would you need in order to turn
  this table into a trading rule, and why is that much harder than it looks?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Classification applied to the same cycles as the back-test; counts reported; thresholds named constants, any change justified | 1 |
| (b), (c) | Regime table (mean, win rate, worst) and grouped bar chart; best strategy flagged; stock-based reclassification with count of changed cycles | 2 |
| (d) | Regime winners explained from payoffs (covered call in sideways, protective put in bear, stock or covered call in bull, straddle in large moves either way); Nifty-vs-stock difference tied to beta, correlation and idiosyncratic risk; hindsight problem recognised | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q5
BULL_THR, BEAR_THR = +0.02, -0.02

**Your answer (Q5d):**

*Thresholds used (and justification if changed); cycles per regime:*

*Winner in each regime vs what the payoffs predict:*

*Nifty-based vs stock-based classification; role of beta and correlation:*

*From a description to a trading rule — what is missing:*

---
## Q6 · Sensitivity and interpretation · **[3 marks]**

Your base case prices every option at realised volatility, with no costs. Test how much the
conclusions depend on that.

**(a)** Re-run the back-test under each of the following and report one table of **total return**
and **max drawdown** per strategy and scenario, with the change from the base case:

1. **Base** — as in Q2.
2. **Volatility premium** — `vol_premium = 0.03`: options priced three volatility points above
   realised, as implied volatility typically is.
3. **Closer strike** — `call_otm = 0.02` for the covered call.
4. **All-in straddle** — `straddle_sizing = "all_in"`: the whole capital spent on premium.
5. **Transaction costs** — the base case less a round-trip cost of **0.5% of each option premium
   traded** and **10 bps of the stock value traded**, charged every cycle. State how you applied it.

**(b)** Write **at least 150 words** answering all four of the following:
- Which strategies **gain** and which **lose** when options become more expensive (scenario 2), and
  by roughly how much per month? Why is this the most important sensitivity of the five?
- What does scenario 4 tell you about leverage and compounding? Refer to the worst month.
- After costs (scenario 5), does the ranking of the strategies change? Which strategy is the most
  cost-sensitive and why?
- State **three limitations** of this back-test that would make you hesitate before putting real
  money behind its winner, other than those already covered above.

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | All five scenarios run through the same engine by changing arguments, not by copy-paste; one comparison table with changes from base; cost convention stated | 2 |
| (b) | ≥150 words; sign and rough size of the volatility-premium effect for each strategy (sellers gain, buyers lose, ≈ vega × 3 points); leverage and volatility drag under all-in sizing; turnover explains cost sensitivity; three further limitations (short sample, model prices, lot size and strike grid, liquidity, early assignment not relevant but physical settlement is, taxes, single stock) | 3 |

In [ ]:
# ---- YOUR CODE HERE ----  Q6 (a)
OPT_COST   = 0.005      # round-trip cost as a fraction of each option premium traded
STOCK_COST = 0.0010     # round-trip cost as a fraction of stock value traded

**Your answer (Q6b) — at least 150 words:**

*How transaction costs were applied:*

*Effect of the volatility premium on each strategy, and why it matters most:*

*What all-in sizing shows about leverage and compounding:*

*Ranking after costs, and the most cost-sensitive strategy:*

*Three further limitations:*

---
## Declaration and checklist

**Sources, collaborators and any AI assistance used** (be specific — which tool, for which part,
and what you did with the output). Full Disclosure of the prompt is expected and costs you nothing; undisclosed use
of AI will be considered as an integrity violation.

*Your declaration:*

---

### Before you submit, confirm every box

- [ ] `NAME`, `CAMPUS_ID`, `TICKER` and `HONOUR` are filled in and the identification cell runs
- [ ] The ticker is the **same** as in your Assignment 03
- [ ] The provided pricer test cell passes **unedited**
- [ ] `Kernel → Restart & Run All` completes with **no errors**
- [ ] Every plot and table is **visible in the saved file** (not an empty output cell)
- [ ] Every `**Your answer:**` cell is filled in — none left blank
- [ ] You stated whether each data series is **live or simulated**, and that option premiums are
      **model-generated**
- [ ] The conventions of Q2 are followed, or every departure is stated and justified
- [ ] Source cited for the dividend yield
- [ ] Q6(b) is at least 150 words
- [ ] Saved as `DRM_Lab05_<CampusID>.ipynb`
- [ ] Uploaded to the Lab 05 assignment on **Nalanda** before the deadline